In [11]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv()
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
minio_url = "http://" + os.getenv("S3_API_ENDPOINT")


minio_client = boto3.client(
    "s3",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=minio_url
)

new_bucket = "exploitation-zone-task2-images-text"
try:
    minio_client.create_bucket(Bucket=new_bucket)
except (minio_client.exceptions.BucketAlreadyExists, minio_client.exceptions.BucketAlreadyOwnedByYou):
    print(f"Bucket '{new_bucket}' already exists")

Bucket 'exploitation-zone-task2-images-text' already exists


In [12]:
def _embed_audio_bytes(audio_bytes):
    buf = io.BytesIO(audio_bytes)
    waveform, sr = torchaudio.load(buf)
    if sr != 48000:
        waveform = torchaudio.functional.resample(waveform, sr, 48000)
    emb = clap.get_audio_embedding_from_data(x=waveform, use_tensor=False)
    return emb.squeeze().tolist()

In [17]:
# ==========================================================
# 🔧 CONFIGURACIÓN E IMPORTS
# ==========================================================
import os
import io
from PIL import Image
import torchaudio
import torch
from imagebind import data, model
import chromadb

# ==========================================================
# 🧠 CARGAR MODELO IMAGEBIND
# ==========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔹 Cargando ImageBind en {device}...")

imagebind_model = model.imagebind_huge(pretrained=True)
imagebind_model.eval().to(device)

print("✅ Modelo ImageBind cargado correctamente.")

# ==========================================================
# 🗄️ CONFIGURACIÓN DE MINIO Y CHROMADB
# ==========================================================

client = chromadb.HttpClient(host="localhost", port=8000)

collection_name = "multimodal_collection_unificada"

# Borrar colección anterior si existe
try:
    client.delete_collection(name=collection_name)
except Exception:
    pass

col = client.get_or_create_collection(name=collection_name)

trusted_zone = "trusted-zone"
exploitation_zone = "exploitation-zone-task2-images-text"

# ==========================================================
# 🔹 FUNCIONES DE EMBEDDING
# ==========================================================
def embed_text(texts):
    inputs = {"text": data.load_and_transform_text(texts)}
    with torch.no_grad():
        emb = imagebind_model(inputs)["text"]
    return emb.cpu().numpy()

def embed_image(pil_images):
    tmp_paths = []
    for i, img in enumerate(pil_images):
        path = f"/tmp/temp_image_{i}.jpg"
        img.save(path)
        tmp_paths.append(path)
    inputs = {"image": data.load_and_transform_vision_data(tmp_paths)}
    with torch.no_grad():
        emb = imagebind_model(inputs)["image"]
    return emb.cpu().numpy()

def embed_audio_bytes(audio_bytes_list):
    tmp_paths = []
    for i, audio_bytes in enumerate(audio_bytes_list):
        path = f"/tmp/temp_audio_{i}.wav"
        with open(path, "wb") as f:
            f.write(audio_bytes)
        tmp_paths.append(path)
    inputs = {"audio": data.load_and_transform_audio_data(tmp_paths)}
    with torch.no_grad():
        emb = imagebind_model(inputs)["audio"]
    return emb.cpu().numpy()

# ==========================================================
# 🖼️ AÑADIR IMÁGENES
# ==========================================================
for obj in minio_client.list_objects(trusted_zone, prefix="images/", recursive=True):
    key = obj.object_name
    response = minio_client.get_object(trusted_zone, key)
    img_data = response.read()
    response.close()
    response.release_conn()

    image = Image.open(io.BytesIO(img_data)).convert("RGB")
    emb = embed_image([image])[0]

    col.add(
        ids=[key],
        embeddings=[emb.tolist()],
        metadatas=[{"bucket": trusted_zone, "key": key, "type": "image"}],
    )

    minio_client.copy_object(
        exploitation_zone,
        key,
        f"/{trusted_zone}/{key}"
    )

# ==========================================================
# 📄 AÑADIR TEXTOS
# ==========================================================
for obj in minio_client.list_objects(trusted_zone, prefix="text/", recursive=True):
    key = obj.object_name
    response = minio_client.get_object(trusted_zone, key)
    text_content = response.read().decode("utf-8")
    response.close()
    response.release_conn()

    emb = embed_text([text_content])[0]

    col.add(
        ids=[key],
        embeddings=[emb.tolist()],
        metadatas=[{"bucket": trusted_zone, "key": key, "type": "text"}],
    )

    minio_client.copy_object(
        exploitation_zone,
        key,
        f"/{trusted_zone}/{key}"
    )

# ==========================================================
# 🎶 AÑADIR AUDIOS
# ==========================================================
for obj in minio_client.list_objects(trusted_zone, prefix="audio/", recursive=True):
    key = obj.object_name
    response = minio_client.get_object(trusted_zone, key)
    audio_bytes = response.read()
    response.close()
    response.release_conn()

    emb = embed_audio_bytes([audio_bytes])[0]

    col.add(
        ids=[key],
        embeddings=[emb.tolist()],
        metadatas=[{"bucket": trusted_zone, "key": key, "type": "audio"}],
    )

    minio_client.copy_object(
        exploitation_zone,
        key,
        f"/{trusted_zone}/{key}"
    )

# ==========================================================
# ✅ RESULTADO FINAL
# ==========================================================
print(f"✅ Total de elementos almacenados: {len(col.get()['ids'])}")
print("🎯 Ahora texto, imagen y audio están en el MISMO espacio vectorial (ImageBind).")


ModuleNotFoundError: No module named 'imagebind'